In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np
import datetime
import matplotlib.pyplot as plt
import itertools
import copy
import numpy as np
from scipy.stats import stats
import random
import os
import sys

In [ ]:
import pickle
from mlxtend.preprocessing import TransactionEncoder

def get_non_binary_light_switches(dataset):
  if dataset == 'hh102':
    return ['L001', 'L005']
  elif dataset == 'hh106':
    return []
  elif dataset == 'hh108':
    return ['L005', 'L002']
  elif dataset == 'hh113':
    return []
  elif dataset == 'hh114':
    return ['L001']

#save data to loc_file
def save_data(dataset, loc_file, pre_loc_file_url = '/content/drive/My Drive/4_Proposed_Approach/'):
  loc_file = pre_loc_file_url + loc_file
  with open(loc_file, 'wb') as filehandle:
      pickle.dump(dataset, filehandle)

#load data from loc_file
def load_data(loc_file, pre_loc_file_url = '/content/drive/My Drive/4_Proposed_Approach/'):
  loc_file = pre_loc_file_url + loc_file
  print(loc_file)
  with open(loc_file, 'rb') as filehandle:
      dataset = pickle.load(filehandle)
  return dataset

def remove_subset(list_of_list):
    sets={frozenset(e) for e in list_of_list}
    us=[]
    while sets:
        e=sets.pop()
        if any(e.issubset(s) for s in sets) or any(e.issubset(s) for s in us):
            continue
        else:
            us.append(sorted(list(e)))
    return us

def convert_transaction_to_df_one_hot_vector(_transaction_dataset, sparse_format = True):
  te = TransactionEncoder()
  if(sparse_format):
    te_ary = te.fit(_transaction_dataset).transform(_transaction_dataset, sparse=True)
    return pd.DataFrame.sparse.from_spmatrix(te_ary, columns=te.columns_)
  te_ary = te.fit(_transaction_dataset).transform(_transaction_dataset)
  return pd.DataFrame(te_ary, columns=te.columns_)

def print_list_with_minK(list_of_list, min_k_item):
  count = 0
  for item in list_of_list:
    if(len(item) >= min_k_item):
      count +=1
      print(item)
  print(f"Len -origin:{len(list_of_list)} ")
  print(f"Len -filter by min_k_item={min_k_item} :{count} ")

def is_binary_device_by_df(df, non_binary_light_switches):
    # non_binary_light_switches = get_non_binary_light_switches(dataset)
    if (df['device_id'][0:1] == 'M'
        or df['device_id'][0:1] == 'D'
        or df['device_id'][0:2] == 'L0'
        and df['device_id'] not in non_binary_light_switches):
        val = bool(True)
    else:
        val = bool(False)
    return val

def is_not_binary_device_by_id(device_id):
    if (device_id[0:1] == 'M'
        or device_id[0:1] == 'D'
        or device_id[0:2] == 'L0'):
        return False
    else:
        return True

In [ ]:
def initial_data_preprocessing(data, dataset):

  data['device_value'] =  data['device_value'].replace(['ON', 'OPEN','OK'], 1)
  data['device_value'] =  data['device_value'].replace(['OFF', 'CLOSE'], 0)

  data.device_value = pd.to_numeric(data.device_value, errors='raise')
  non_binary_light_switches = get_non_binary_light_switches(dataset)

  data['is_binary_device'] = data.apply(is_binary_device_by_df, axis=1, non_binary_light_switches=non_binary_light_switches)

  data.drop(data[data['device_id'].astype(str).str[0:2] == 'BA'].index, axis =0, inplace = True)

  data.drop(['ann'], axis=1, inplace =True)
  data.drop(['location_1'], axis=1, inplace =True)
  data.drop(['location_2'], axis=1, inplace =True)
  items = data['device_id'].unique()


**Convert data to correct format**



In [ ]:
# filename = '/content/drive/My Drive/0_Original_Datasets/hh102/hh102.rawdata.txt'
filename = '/content/drive/My Drive/0_Original_Datasets/hh106/hh106.rawdata.txt'
# filename = '/content/drive/My Drive/0_Original_Datasets/hh108/hh108.rawdata.txt'
# filename = '/content/drive/My Drive/0_Original_Datasets/hh113/hh113.rawdata.txt'
# filename = '/content/drive/My Drive/0_Original_Datasets/hh114/hh114.rawdata.txt'

data_raw = pd.read_csv(filename, sep ='\t', names = ["datetime", "device_id", "location_1", "location_2", "device_value", "ann"])
                       #parse_dates ={'datetime': ['date', 'time']} )

In [ ]:
from enum import Enum

class SensorEvent:
  def __init__(self, timestamp, device_id, location, obj, device_value, ann):
    self.timestamp = timestamp
    self.device_id = device_id
    self.location = location
    self.obj = obj
    self.device_value = device_value
    self.ann = ann

  def __str__(self):
    return f'Sensor event(timestamp = {self.timestamp}, device id = {self.device_id}, location = {self.location}, obj = {self.obj}, device value = {self.device_value}), is generated = {self.is_generated}'

class ProcessedSensorEvents:
  def __init__(self, datetime, device_id, device_value, is_binary_device):
    self.datetime = datetime
    self.device_id = device_id
    self.device_value = device_value
    self.is_binary_device = is_binary_device

In [ ]:
from matplotlib import pyplot as plt

def convert_sensor_event_to_list(event):
  return [event.timestamp, event.device_id, event.location, event.obj, event.device_value, event.ann]

def preprocessing(data_raw, dataset, training_data_name, value_threshold, start_time_jump, training_weeks, rate):
  data_raw['datetime'] = data_raw['datetime'].apply(lambda x: datetime.datetime.strptime(x[:19], '%Y-%m-%d %H:%M:%S'))
  device_list = data_raw['device_id'].unique()

  non_binary_devices = list(filter(is_not_binary_device_by_id, device_list))
  device_value_map = {key: -1 for key in non_binary_devices}

  start_date = data_raw.iloc[0].datetime + datetime.timedelta(weeks=start_time_jump)
  end_date = start_date + datetime.timedelta(weeks=training_weeks)

  if training_weeks == 0:
    data = data_raw
  else:
    data = data_raw[data_raw['datetime'].between(start_date, end_date)]

  sensor_events = list(map(lambda x: SensorEvent(x[0], x[1], x[2], x[3], x[4], x[5]), data_raw.values.tolist()))
  filtered_events = []

  for index, event in enumerate(sensor_events):
    device = event.device_id
    print(index)

    if device not in non_binary_devices:
      filtered_events.append(event)
      continue

    device_value = float(event.device_value)

    if device_value_map[device] == -1:
      filtered_events.append(event)
      device_value_map[device] = device_value
      continue

    if abs(device_value - device_value_map[device]) > value_threshold:
      filtered_events.append(event)
      device_value_map[device] = device_value

  test_data_rows = map(convert_sensor_event_to_list, filtered_events)
  test_df = pd.DataFrame(test_data_rows, columns=['datetime', 'device_id', 'location_1', 'location_2', 'device_value', 'ann'])

  initial_data_preprocessing(test_df, dataset)

  frame_size = test_df.shape[0]
  training_size = round(frame_size*rate)

  save_data(test_df[0:training_size], training_data_name)

def check_light_sensor_change(data, sensor):
  value_change_set = set()
  sensor_values = data[data['device_id'] == sensor]['device_value'].tolist()

  for i in range(len(sensor_values) - 1):
    value_change_set.add(float(sensor_values[i+1]) - float(sensor_values[i]))

  value_change_set = list(value_change_set)
  value_change_set.sort()

  plt.figure()
  plt.plot(value_change_set, marker='o')
  plt.show()

def convert_to_processed_sensor_events(data):
  raw_data = data.values.tolist()
  return list(map(lambda x: ProcessedSensorEvents(x[0], x[1], x[2], x[3]), raw_data))

def process_abnormal_binary_devices(data):
  binary_devices = data.loc[(data['is_binary_device'] == True)]['device_id'].unique()
  sensor_events = convert_to_processed_sensor_events(data)
  abnormal_binary_devices = []

  for device in binary_devices:
    device_values = data[data['device_id'] == device]['device_value'].unique()
    if len(device_values) != 2:
      abnormal_binary_devices.append(device)
  print(abnormal_binary_devices)

  binary_device_value = {key:-1 for key in abnormal_binary_devices}
  binary_device_status = {key:-1 for key in abnormal_binary_devices}
  processed_events = []

  for i in range(len(sensor_events)):
    event = sensor_events[i]
    if event.device_id not in abnormal_binary_devices:
      processed_events.append(event)
    else:
      if binary_device_value[event.device_id] == -1:
        binary_device_value[event.device_id] = float(event.device_value)
        continue

      datetime = event.datetime
      device_id = event.device_id

      if float(event.device_value) < binary_device_value[device_id] and binary_device_status[device_id] in [-1, 100]:
        device_value = 0
        processed_events.append(ProcessedSensorEvents(datetime, device_id, device_value, True))
        binary_device_status[device_id] = 0

      if float(event.device_value) > binary_device_value[device_id] and binary_device_status[device_id] in [-1, 0]:
        device_value = 100
        processed_events.append(ProcessedSensorEvents(datetime, device_id, device_value, True))
        binary_device_status[device_id] = 100

      binary_device_value[event.device_id] = float(event.device_value)

  def convert_processed_event_to_list(event):
    return [event.datetime, event.device_id, event.device_value, event.is_binary_device]

  data_rows = map(convert_processed_event_to_list, processed_events)
  processed_df = pd.DataFrame(data_rows, columns=['datetime', 'device_id', 'device_value', 'is_binary_device'])
  return processed_df

In [ ]:
preprocessing(data_raw, 'hh106', 'hh106_training.data', 2, 0, 0, 0.9)

KeyboardInterrupt: ignored

In [ ]:
data_raw.drop(data_raw[(data_raw['ann'] == 'Control4-Button') | (data_raw['ann'] == 'Control4-Radio')].index, inplace = True)
data = data_raw.copy(deep=True)
initial_data_preprocessing(data, 'hh102')
# data['datetime'] = data['datetime'].apply(lambda x: datetime.datetime.strptime(x[:19], '%Y-%m-%d %H:%M:%S'))

In [ ]:
data[data['device_id'] == 'L003']

,datetime,device_id,location_1,location_2,device_value,ann,is_binary_device
846,2011-06-15 09:48:56.425647,L003,Bathroom,Bathroom,100,Control4-Light,True
1020,2011-06-15 09:55:40.643635,L003,Bathroom,Bathroom,0,Control4-Light,True
1578,2011-06-15 12:09:27.512866,L003,Bathroom,Bathroom,100,Control4-Light,True
1942,2011-06-15 12:58:42.289148,L003,Bathroom,Bathroom,0,Control4-Light,True
3114,2011-06-15 16:08:16.031180,L003,Bathroom,Bathroom,100,Control4-Light,True
...,...,...,...,...,...,...,...
6423596,2014-03-14 12:01:17.096111,L003,Bathroom,Bathroom,100,Control4-Light,True
6423597,2014-03-14 12:01:35.918640,L003,Bathroom,Bathroom,0,Control4-Light,True
6425422,2014-03-16 15:14:49.522854,L003,Bathroom,Bathroom,100,Control4-Light,True
6425430,2014-03-16 15:17:22.788983,L003,Bathroom,Bathroom,0,Control4-Light,True


In [ ]:
data[data['device_id'] == 'L001']

In [ ]:
start = data.iloc[0].datetime
end = start + datetime.timedelta(weeks=60)
# data.loc[(data['device_id'] == 'D001') & (data['datetime'].between(start, end))]

**Split data set into training and testing set**

In [ ]:
rate = 0.7
start_time_jump = 0
frame_size = data.shape[0]
training_size = round(frame_size*rate)
start_date = data.iloc[0].datetime + datetime.timedelta(weeks=start_time_jump)
end_date = start_date + datetime.timedelta(weeks=53)
data_training = data[data['datetime'].between(start_date, end_date)]
# data_testing = data[training_size:-1]
# data_training

# save_data(data_training, 'hh103_training.data')
# save_data(data_training, 'hh101_training.data')
save_data(data_training, 'hh102_training.data')
# save_data(data_testing, 'hh103_testing.data')

In [ ]:
data_training['device_id'].unique()

In [ ]:
data = load_data('hh102_training.data')

In [ ]:
data[data['device_id'] == 'L005']

,datetime,device_id,device_value,is_binary_device
4995,2011-06-15 22:43:53,L005,2,False
4996,2011-06-15 22:43:53,L005,42,False
4997,2011-06-15 22:43:54,L005,100,False
5013,2011-06-15 22:44:22,L005,99,False
5014,2011-06-15 22:44:23,L005,89,False
...,...,...,...,...
4600779,2013-12-21 05:44:01,L005,26,False
4600780,2013-12-21 05:44:02,L005,12,False
4600781,2013-12-21 05:44:05,L005,3,False
4600782,2013-12-21 05:44:06,L005,2,False
